# 05 Fältdata från riktiga besökare

Webperf testar webbplatserna med automatiska verktyg från en och samma plats. Det kallas labbdata. Chrome UX Report, förkortat CrUX, mäter i stället hur snabbt sidorna faktiskt laddar för riktiga besökare som använder Chrome. Det kallas fältdata. Genom serien jämför vi de två mätmetoderna.

CrUX räknar samman de senaste 28 dagarna. Vi hämtar tre mått för varje webbplats.

- **LCP**, Largest Contentful Paint, är tiden i millisekunder tills det största innehållet på sidan syns.
- **INP**, Interaction to Next Paint, är tiden i millisekunder från att besökaren klickar eller trycker tills sidan visar en reaktion.
- **CLS**, Cumulative Layout Shift, mäter hur mycket innehållet flyttar sig medan sidan laddar. Det är ett tal utan enhet.

För varje mått använder vi den sjuttiofemte percentilen, p75. Det betyder att tre av fyra sidvisningar var lika bra eller bättre än värdet.

In [ ]:
import json
import os
import time
from pathlib import Path
from urllib.parse import urlsplit

import httpx
import pandas as pd
from dotenv import load_dotenv

## API-nyckeln

CrUX kräver en kostnadsfri API-nyckel från Google. Nyckeln ligger i filen `python/.env`, som git ignorerar, så att den aldrig hamnar i en notebook eller i repot. Filen innehåller en rad.

```
CRUX_API_KEY=din-nyckel
```

`load_dotenv` läser filen och lägger nyckeln bland miljövariablerna. Därifrån hämtar vi den med `os.environ.get`. Vi skriver aldrig ut själva nyckeln.

In [ ]:
load_dotenv(Path("..") / ".env")
NYCKEL = os.environ.get("CRUX_API_KEY")

if NYCKEL:
    print("Nyckeln finns")
else:
    raise RuntimeError("CRUX_API_KEY saknas. Lägg den i python/.env enligt README och starta om kerneln.")

## Från adress till origin

CrUX samlar data per origin, alltså protokoll och domännamn utan sökväg, till exempel `https://ale.se`. Några av Webperfs adresser pekar på en undersida, så vi tar fram originet ur adressen.

`urlsplit` delar en adress i sina delar. `scheme` är protokollet och `netloc` domännamnet.

In [ ]:
delar = urlsplit("https://www.hoganas.se/Invanare/")

delar.scheme, delar.netloc, delar.path

Vi gör en liten funktion av det och använder den på alla Webperfs adresser. Först läser vi in Webperfs data som vi sparade i notebook 03, på samma sätt som i notebook 04. Med `map` används funktionen på varje värde i kolumnen.

In [ ]:
RADATA = Path("..") / "data" / "radata"

kommuner_fil = json.loads((RADATA / "webperf-kommuner.json").read_text(encoding="utf-8"))
regioner_fil = json.loads((RADATA / "webperf-regioner.json").read_text(encoding="utf-8"))

kommuner = pd.json_normalize(kommuner_fil["svar"]["municipalities"])
regioner = pd.json_normalize(regioner_fil["svar"]["regions"])
kommuner["typ"] = "kommun"
regioner["typ"] = "region"

webperf = pd.concat([kommuner, regioner], ignore_index=True)
webperf = webperf[["id", "title", "typ", "website"]]


def till_origin(adress):
    delar = urlsplit(adress)
    return f"{delar.scheme}://{delar.netloc}"


webperf["origin"] = webperf["website"].map(till_origin)

webperf.head()

## Fråga CrUX

Vi frågar CrUX genom att skicka en POST med originet och de mått vi vill ha. `json=` gör om ordboken till JSON åt oss. Nyckeln skickar vi i headern `X-Goog-Api-Key` i stället för i adressen. Då syns den inte om ett felmeddelande skriver ut adressen.

Om CrUX saknar data för ett origin blir svaret statuskod 404. Det är inget fel i vår kod, så funktionen avbryter inte då. Den lämnar tillbaka statuskoden och svaret, så får vi själva avgöra vad vi gör.

In [ ]:
CRUX = "https://chromeuxreport.googleapis.com/v1/records:queryRecord"
MATT = ["largest_contentful_paint", "interaction_to_next_paint", "cumulative_layout_shift"]


def fraga_crux(origin):
    svar = httpx.post(
        CRUX,
        json={"origin": origin, "metrics": MATT},
        headers={"X-Goog-Api-Key": NYCKEL},
        timeout=30,
    )
    if svar.status_code not in (200, 404):
        svar.raise_for_status()
    return svar.status_code, svar.json()

Vi provar med en webbplats.

In [ ]:
status, ale = fraga_crux("https://ale.se")

status, list(ale["record"].keys())

Svaret har tre delar. `key` är originet vi frågade om, `metrics` innehåller måtten och `collectionPeriod` anger vilka dagar datan gäller.

In [ ]:
ale["record"]["collectionPeriod"]

Varje mått har ett histogram, som visar hur stor andel av sidvisningarna som var bra, behövde förbättras eller var dåliga, och percentilen p75.

In [ ]:
ale["record"]["metrics"]["largest_contentful_paint"]

In [ ]:
for matt in MATT:
    p75 = ale["record"]["metrics"][matt]["percentiles"]["p75"]
    print(matt, repr(p75), type(p75).__name__)

LCP och INP är heltal, men CLS är text. Det står också i dokumentationen för CrUX. Vi behöver alltså göra om CLS till ett tal innan vi kan räkna med det. `repr` visar värdet med citattecken när det är text, så att skillnaden syns.

## Alla webbplatser

Nu frågar vi om alla origin. CrUX tillåter 150 frågor per minut. Vi väntar en halv sekund mellan frågorna och håller oss därmed under gränsen. Det tar ett par minuter.

Svaren sparar vi i en ordbok med originet som nyckel.

In [ ]:
crux_svar = {}

for nummer, origin in enumerate(webperf["origin"], start=1):
    crux_svar[origin] = fraga_crux(origin)
    if nummer % 50 == 0:
        print(nummer, "klara")
    time.sleep(0.5)

len(crux_svar)

`enumerate` numrerar varje varv i loopen och `%` ger resten vid division. `nummer % 50 == 0` är alltså sant vart femtionde varv, så att vi ser att loopen går framåt.

Hur många fick data?

In [ ]:
pd.Series([status for status, _ in crux_svar.values()]).value_counts()

## När CrUX saknar data

Vi tar fram statuskoden för varje webbplats och listar de som fick 404.

In [ ]:
status_per_origin = {origin: status for origin, (status, _) in crux_svar.items()}
webperf["crux_status"] = webperf["origin"].map(status_per_origin)

webperf.loc[webperf["crux_status"] == 404, ["title", "origin"]]

CrUX tar bara med origin som har tillräckligt många besökare och vars startsida är offentligt sökbar. En startsida som svarar med något annat än statuskod 200, eller som säger åt sökmotorer att inte indexera den med `noindex`, gör att hela originet saknas i CrUX. Det beskrivs i [CrUX metodbeskrivning](https://developer.chrome.com/docs/crux/methodology).

Ett tredje skäl är att besökarna hamnar på ett annat origin än det vi frågar om. Svarar en webbplats både med och utan `www.` kan CrUX ha data för bara det ena. Då är det värt att kontrollera vilken adress som webbplatsen själv använder.

## Från svar till tabell

Vi skriver en funktion som plockar ut det vi behöver ur ett svar. Saknas ett mått, vilket händer när för få besökare har mätts, blir värdet `None`. Metoden `get` på en ordbok ger `None` i stället för ett fel när nyckeln saknas.

In [ ]:
def plocka_ut(svar):
    matt = svar["record"]["metrics"]
    period = svar["record"]["collectionPeriod"]

    def p75(namn):
        if namn in matt:
            return matt[namn]["percentiles"]["p75"]
        return None

    return {
        "lcp_p75": p75("largest_contentful_paint"),
        "inp_p75": p75("interaction_to_next_paint"),
        "cls_p75": p75("cumulative_layout_shift"),
        "period_start": "{year}-{month:02d}-{day:02d}".format(**period["firstDate"]),
        "period_slut": "{year}-{month:02d}-{day:02d}".format(**period["lastDate"]),
    }


plocka_ut(ale)

`{month:02d}` skriver månaden med två siffror, så att september blir `09`. `**period["firstDate"]` lämnar över år, månad och dag som namngivna värden till `format`.

Nu går vi igenom alla svar med statuskod 200 och bygger en tabell med en rad per origin.

In [ ]:
rader = []
for origin, (status, svar) in crux_svar.items():
    if status == 200:
        rad = plocka_ut(svar)
        rad["origin"] = origin
        rader.append(rad)

crux = pd.DataFrame(rader)

crux.dtypes

CLS är fortfarande text. Med `pd.to_numeric` gör vi om den till tal. LCP och INP blir `Int64`, heltal som kan sakna värde.

In [ ]:
crux["cls_p75"] = pd.to_numeric(crux["cls_p75"])
crux["lcp_p75"] = crux["lcp_p75"].astype("Int64")
crux["inp_p75"] = crux["inp_p75"].astype("Int64")

crux.dtypes

## Koppla till Webperfs webbplatser

Vi kopplar ihop tabellen med Webperfs webbplatser via `origin` och kontrollerar att antalet rader inte ändras.

In [ ]:
falt = webperf.merge(crux, on="origin", how="left")

len(falt) == len(webperf)

In [ ]:
falt[["title", "origin", "lcp_p75", "inp_p75", "cls_p75"]].head()

Hur många saknar varje mått?

In [ ]:
falt[["lcp_p75", "inp_p75", "cls_p75"]].isna().sum()

Webbplatser som saknas helt i CrUX saknar alla tre måtten. Några webbplatser har data för CLS men inte för LCP eller INP. Varje mått kräver tillräckligt många mätningar, och INP mäts bara när besökaren interagerar med sidan.

Alla svar ska gälla samma period, annars jämför vi olika tider.

In [ ]:
falt[["period_start", "period_slut"]].value_counts()

## Spara rådata

Vi sparar alla svar från CrUX som de kom, med statuskoden, tillsammans med vilka mått vi frågade efter och när vi hämtade dem. Nyckeln finns inte i svaren och sparas inte.

In [ ]:
hamtad = pd.Timestamp.now(tz="Europe/Stockholm").isoformat(timespec="seconds")

paket = {
    "hamtad": hamtad,
    "kalla": CRUX,
    "matt": MATT,
    "svar": {origin: {"status": status, "svar": svar} for origin, (status, svar) in crux_svar.items()},
}

fil = RADATA / "crux.json"
fil.write_text(json.dumps(paket, ensure_ascii=False, indent=2), encoding="utf-8")
fil

## Det här vet vi nu

- CrUX ger fältdata från riktiga besökare i Chrome, samlade över 28 dagar.
- Data finns per origin, och originet behöver vara det som besökarna faktiskt hamnar på.
- En del webbplatser saknar data helt, till exempel för att de har för få besökare eller för att startsidan inte får indexeras.
- Enskilda mått kan saknas även när andra finns.
- CLS kommer som text och behöver göras om till tal.

I nästa notebook tvättar vi datan från alla tre källorna så att den går att lita på och jämföra.